In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.functions import current_date, datediff

# ============================================================
# FACT_INVENTORY
# ============================================================
silver_inventory = spark.table("pharma_catalog.silver.silver_inventory")
dim_product      = spark.table("pharma_catalog.gold.dim_product")

fact_inventory = silver_inventory \
    .join(dim_product.select("product_id", "unit_price"), on="product_id", how="left") \
    .withColumn("stock_value",    F.round(F.col("quantity_on_hand") * F.col("unit_price"), 2)) \
    .withColumn("days_to_expiry", datediff(F.col("expiry_date"), current_date())) \
    .withColumn("is_expired",     F.when(F.col("days_to_expiry") < 0, True).otherwise(False)) \
    .select(
        "inventory_id",
        "product_id",
        "warehouse_id",
        "batch_id",
        "quantity_on_hand",
        "manufacturing_date",
        "expiry_date",
        "last_inventory_update",
        "stock_value",
        "days_to_expiry",
        "is_expired"
    )

fact_inventory.write.mode("overwrite").saveAsTable("pharma_catalog.gold.fact_inventory")
print(f"✅ fact_inventory : {fact_inventory.count()} lignes")

# COMMAND ----------
# ============================================================
# FACT_SALES_ORDERS
# ============================================================
fact_sales_orders = spark.table("pharma_catalog.silver.silver_sales_orders") \
    .withColumn("service_level",  F.round(F.col("quantity_delivered") / F.col("quantity_ordered"), 4)) \
    .withColumn("quantity_gap",   F.col("quantity_ordered") - F.col("quantity_delivered")) \
    .withColumn("is_complete",    F.when(F.col("quantity_gap") == 0, True).otherwise(False)) \
    .select(
        "sales_order_id",
        "order_date",
        "product_id",
        "warehouse_id",
        "customer_type",
        "quantity_ordered",
        "quantity_delivered",
        "sales_amount",
        "service_level",
        "quantity_gap",
        "is_complete"
    )

fact_sales_orders.write.mode("overwrite").saveAsTable("pharma_catalog.gold.fact_sales_orders")
print(f"✅ fact_sales_orders : {fact_sales_orders.count()} lignes")

# COMMAND ----------
# ============================================================
# FACT_SUPPLIER_ORDERS
# ============================================================
fact_supplier_orders = spark.table("pharma_catalog.silver.silver_supplier_orders") \
    .withColumn("delay_days", 
        F.when(
            F.col("actual_delivery_date").isNotNull(),
            datediff(F.col("actual_delivery_date"), F.col("expected_delivery_date"))
        ).otherwise(F.lit(None))
    ) \
    .withColumn("is_late",   
        F.when(F.col("delay_days") > 0, True).otherwise(False)
    ) \
    .withColumn("fill_rate", 
        F.round(F.col("quantity_received") / F.col("quantity_ordered"), 4)
    ) \
    .select(
        "supplier_order_id",
        "supplier_id",
        "product_id",
        "warehouse_id",
        "order_date",
        "expected_delivery_date",
        "actual_delivery_date",
        "quantity_ordered",
        "quantity_received",
        "order_status",
        "delay_days",
        "is_late",
        "fill_rate"
    )

fact_supplier_orders.write.mode("overwrite").saveAsTable("pharma_catalog.gold.fact_supplier_orders")
print(f"✅ fact_supplier_orders : {fact_supplier_orders.count()} lignes")

# COMMAND ----------
# Vérification finale
print("=== GOLD FACTS ===")
for table in ["fact_inventory", "fact_sales_orders", "fact_supplier_orders"]:
    count = spark.table(f"pharma_catalog.gold.{table}").count()
    print(f"  {table} : {count} lignes")